In [3]:
import numpy as np
import pandas as pd

In [22]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score

##Display pipeline

from sklearn import set_config
set_config(display='diagram')

In [5]:
P_records = pd.read_csv('patient_records.csv')
P_records.head(2)

,patient_id,first_name,last_name,gender,age,blood_type,state,smoking_status,insurance_type,primary_condition,height_cm,weight_kg,systolic_bp,diastolic_bp,heart_rate_bpm,last_admission_date
0,PT100000,Sarah,Jackson,Male,0,O+,CA,Former,Private,Type 2 Diabetes,152.3,81.8,71,58,75,2024-07-19
1,PT100001,Jessica,Smith,Female,53,O+,PA,Former,Medicaid,Anxiety Disorder,183.2,45.6,114,89,75,2025-01-16


In [6]:
P_records.isnull().sum()

patient_id              0
first_name              0
last_name               0
gender                  0
age                     0
blood_type             55
state                   0
smoking_status         52
insurance_type          0
primary_condition       5
height_cm              53
weight_kg              48
systolic_bp             0
diastolic_bp            0
heart_rate_bpm          0
last_admission_date     0
dtype: int64

In [7]:
P_records.drop(columns=['patient_id', 'first_name', 'last_name', 'gender', 'insurance_type', 'primary_condition', 'last_admission_date'], inplace=True)

In [8]:
X=P_records[['age', 'blood_type', 'smoking_status', 'height_cm', 'weight_kg', 'systolic_bp', 'diastolic_bp']]
X

,age,blood_type,smoking_status,height_cm,weight_kg,systolic_bp,diastolic_bp
0,0,O+,Former,152.3,81.8,71,58
1,53,O+,Former,183.2,45.6,114,89
2,70,A+,Never,178.1,66.5,104,71
3,52,O+,Former,171.5,NaN,147,77
4,36,O+,Never,169.7,65.0,117,59
...,...,...,...,...,...,...,...
1195,27,NaN,Never,177.1,73.5,136,62
1196,32,O+,Never,175.9,83.8,146,73
1197,52,A+,Current,171.0,66.0,132,94
1198,27,O-,Never,175.2,NaN,157,71


In [9]:
Y=P_records['heart_rate_bpm']
Y

0        75
1        75
2        66
3        99
4        72
       ... 
1195     87
1196    102
1197     66
1198     94
1199     94
Name: heart_rate_bpm, Length: 1200, dtype: int64

In [10]:
X_train, X_test, Y_train, Y_test=train_test_split(X, Y, test_size=0.4)
X_train, X_test, Y_train, Y_test

(      age blood_type smoking_status  height_cm  weight_kg  systolic_bp  \
 191    35         O+          Never      169.9       73.4          134   
 635    90         O-          Never      188.2      101.1          103   
 643    50        NaN          Never      158.2      125.0          123   
 690    32         O-          Never      165.4       69.0          121   
 22      9         O+          Never      172.2        NaN          164   
 ...   ...        ...            ...        ...        ...          ...   
 608    59         A-          Never      160.8       82.1          119   
 724    41         O-         Former      163.4       84.4          129   
 1062   82         A-          Never      162.4       77.7          140   
 333    20         O-          Never      172.7       93.6          111   
 583    78         A+         Former      169.8       77.3          115   
 
       diastolic_bp  
 191             94  
 635             66  
 643             91  
 690      

In [24]:
### missing values

trs1 = ColumnTransformer([('impute_height_cm', SimpleImputer(), [3]), ('impute_weight_kg', SimpleImputer(), [4]), ('blood_type', SimpleImputer(strategy='most_frequent'), [1]), ('smoking_status', SimpleImputer(strategy='most_frequent'), [2])], remainder='passthrough')

In [17]:
## encoding

trs2 = ColumnTransformer([('ohe_blood_type_smoking_status', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), [1, 2])], remainder='passthrough')

In [18]:
## scaling

trs3=ColumnTransformer([('scale', MinMaxScaler(), slice (0,16))])

In [19]:
## feture selection

trs4 = SelectKBest(score_func=chi2, k=10)

In [20]:
## Logic model

trs5 = DecisionTreeClassifier()

### CReating Pipeline

In [26]:
## Pipeline

pipe = Pipeline([('trs1', trs1), 
                ('trs2', trs2), 
                ('trs3', trs3), 
                ('trs4', trs4), 
                ('trs5', trs5)
                ])

### can also use make_pipeline

In [ ]:
## Alternative

pipe2 = make_pipeline(trs1, trs2, trs3, trs4, trs5)

In [27]:
pipe.fit(X_train, Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trs1', ...), ('trs2', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](65,)","[ 28, 34, 41,...,105,107,109]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['age','blood_type','smoking_status',...,'weight_kg','systolic_bp', 'diastolic_bp']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('impute_height_cm', ...), ('impute_weight_kg', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default 